# COLAB_ROME_GPT2_FULL_EXPANDED

Full-evaluation notebook (CrowS full + BBQ full) with expanded edits (120/180).

In [ ]:
# Purpose: Mount Google Drive to access project files.

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Purpose: Initialize Colab and install dependencies for GPT-2 editing pipeline.

%cd /content
!rm -rf /content/EasyEdit /content/work
!git clone --depth 1 https://github.com/zjunlp/EasyEdit.git

!pip -q install -U pip setuptools wheel
!pip -q install "PyYAML==6.0.3"
!pip -q install "datasets==2.20.0"
!pip -q install "transformers==4.57.1" "tokenizers==0.22.1" "sentence-transformers==3.2.1"
!pip -q install accelerate sentencepiece tqdm pandas numpy requests matplotlib hydra-core omegaconf einops higher iopath fairscale timm peft
!pip -q install "rouge==1.0.1" "av==14.2.0" qwen_vl_utils zhipuai "pyjwt==2.8.0"


In [ ]:
# Purpose: Copy scripts and data from your Drive project into /content/work.

import os, shutil, glob

DST = '/content/work'
os.makedirs(DST, exist_ok=True)

candidates = glob.glob('/content/drive/MyDrive/**/scripts/run_edit_fairness_rounds.py', recursive=True)
if not candidates:
    raise FileNotFoundError('scripts/run_edit_fairness_rounds.py not found in MyDrive; please sync your project first')

src_script = candidates[0]
ROOT = os.path.dirname(os.path.dirname(src_script))
print('Detected ROOT =', ROOT)

required_scripts = [
    'scripts/run_edit_fairness_rounds.py',
    'scripts/run_fairness_eval.py',
    'scripts/calc_drift_metrics.py',
    'scripts/plot_drift.py',
]

for rel in required_scripts:
    src = os.path.join(ROOT, rel)
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    shutil.copy2(src, os.path.join(DST, os.path.basename(rel)))

for pattern in ['data/*.json', 'data/*.jsonl']:
    for src in glob.glob(os.path.join(ROOT, pattern)):
        shutil.copy2(src, os.path.join(DST, os.path.basename(src)))

required_data = ['edits_bias_stress_60.json', 'crows_pairs_full.jsonl', 'bbq_full.jsonl']
for fn in required_data:
    p = os.path.join(DST, fn)
    if not os.path.exists(p):
        raise FileNotFoundError(f'missing required data: {p}')

print('copied files in /content/work:', sorted(os.listdir(DST)))


In [ ]:
# Purpose: Patch EasyEdit imports and compatibility issues for stable Colab runs.

from pathlib import Path

# nethook hotfix for with_kwargs=True hook signature in newer PyTorch.
nethook_py = Path('/content/EasyEdit/easyeditor/util/nethook.py')
nt = nethook_py.read_text(encoding='utf-8')
old_sig = "def retain_hook(m, inputs, output, kwargs=None):"
new_sig = "def retain_hook(m, inputs, kwargs, output):"
if old_sig in nt:
    nt = nt.replace(old_sig, new_sig)
old_legacy = """            def legacy_hook(m, inputs, output):
                return retain_hook(m, inputs, output, kwargs=None)
"""
new_legacy = """            def legacy_hook(m, inputs, output):
                return retain_hook(m, inputs, None, output)
"""
if old_legacy in nt:
    nt = nt.replace(old_legacy, new_legacy)
nethook_py.write_text(nt, encoding='utf-8')
print('patched nethook signature')

# Minimize import chain.
init_py = Path('/content/EasyEdit/easyeditor/__init__.py')
init_py.write_text(
    "from .editors.editor import BaseEditor\n"
    "from .models.ft.ft_hparams import FTHyperParams\n"
    "from .models.rome.rome_hparams import ROMEHyperParams\n"
    "from .models.memit.memit_hparams import MEMITHyperParams\n"
    "from .models.mend.mend_hparams import MENDHyperParams\n"
    "from .models.grace.grace_hparams import GraceHyperParams\n"
    "from .models.wise.wise_hparams import WISEHyperParams\n",
    encoding='utf-8'
)

alg_dict_py = Path('/content/EasyEdit/easyeditor/util/alg_dict.py')
alg_dict_py.write_text(
    "from ..models.rome import apply_rome_to_model\n"
    "from ..models.memit import apply_memit_to_model\n"
    "from ..models.mend import MendRewriteExecutor\n"
    "from ..models.ft import apply_ft_to_model\n"
    "from ..models.grace import apply_grace_to_model\n"
    "from ..models.wise import apply_wise_to_model\n\n"
    "ALG_DICT = {\n"
    "    'ROME': apply_rome_to_model,\n"
    "    'MEMIT': apply_memit_to_model,\n"
    "    'MEND': MendRewriteExecutor().apply_to_model,\n"
    "    'FT': apply_ft_to_model,\n"
    "    'GRACE': apply_grace_to_model,\n"
    "    'WISE': apply_wise_to_model,\n"
    "}\n\n"
    "ALG_MULTIMODAL_DICT = {}\n"
    "PER_ALG_DICT = {}\n"
    "DS_DICT = {}\n"
    "MULTIMODAL_DS_DICT = {}\n"
    "PER_DS_DICT = {}\n"
    "Safety_DS_DICT = {}\n",
    encoding='utf-8'
)

models_init = Path('/content/EasyEdit/easyeditor/models/__init__.py')
models_init.write_text(
    "from .ft import *\n"
    "from .rome import *\n"
    "from .memit import *\n"
    "from .mend import *\n"
    "from .grace import *\n"
    "from .wise import *\n",
    encoding='utf-8'
)

editor_py = Path('/content/EasyEdit/easyeditor/editors/editor.py')
txt = editor_py.read_text(encoding='utf-8')
txt = txt.replace('from ..models.melo.melo import LORA', 'LORA = None  # patched: disable melo import')
txt = txt.replace('if isinstance(edited_model, LORA):', 'if (LORA is not None) and isinstance(edited_model, LORA):')
editor_py.write_text(txt, encoding='utf-8')
print('patched imports and editor guard')

# MEMIT dict-output compatibility patch.
compute_z_py = Path('/content/EasyEdit/easyeditor/models/memit/compute_z.py')
cz = compute_z_py.read_text(encoding='utf-8')
old = """        loss_layer_out = tr[hparams.layer_module_tmp.format(loss_layer)].output
        if isinstance(loss_layer_out, (list, tuple)):
            output = loss_layer_out[0]
        else:
            output = loss_layer_out
"""
new = """        loss_layer_out = tr[hparams.layer_module_tmp.format(loss_layer)].output
        if isinstance(loss_layer_out, (list, tuple)):
            output = loss_layer_out[0]
        elif isinstance(loss_layer_out, dict):
            output = None
            for k in ('hidden_states', 'last_hidden_state', 'output'):
                v = loss_layer_out.get(k, None)
                if isinstance(v, torch.Tensor):
                    output = v
                    break
                if isinstance(v, (list, tuple)) and len(v) > 0 and isinstance(v[0], torch.Tensor):
                    output = v[0]
                    break
            if output is None:
                for _, v in loss_layer_out.items():
                    if isinstance(v, torch.Tensor):
                        output = v
                        break
            if output is None:
                raise TypeError(f'Unsupported dict output at loss layer: {list(loss_layer_out.keys())[:10]}')
        else:
            output = loss_layer_out
"""
if old in cz:
    cz = cz.replace(old, new)
    compute_z_py.write_text(cz, encoding='utf-8')
    print('patched MEMIT compute_z dict-output')
else:
    print('MEMIT compute_z patch skipped (pattern not found)')


In [ ]:
# Purpose: Prepare full evaluation datasets and generate expanded edit sets (120/180).

import json
from pathlib import Path

# Validate full eval files.
for fn in ['crows_pairs_full.jsonl', 'bbq_full.jsonl']:
    p = Path('/content/work') / fn
    if not p.exists():
        raise FileNotFoundError(f'missing full evaluation file: {p}')
    n = sum(1 for _ in p.open('r', encoding='utf-8'))
    print(f'{fn}: {n} lines')

base_path = Path('/content/work/edits_bias_stress_60.json')
if not base_path.exists():
    raise FileNotFoundError(f'missing base edit set: {base_path}')
base = json.loads(base_path.read_text(encoding='utf-8'))
print('base edits:', len(base))


def rewrite_with_template(src: str, variant_idx: int) -> str:
    templates = [
        'In recruitment, {}',
        'In many workplaces, {}',
        'According to this view, {}',
        'For this role, {}',
        'In this context, {}',
    ]
    core = src[0].lower() + src[1:] if src.startswith('The ') else src
    return templates[variant_idx % len(templates)].format(core)


def build_rephrase(src: str) -> str:
    return src.replace('The ', '').replace(' is', ' tends to be')


def build_records(base_records, size: int):
    if size < len(base_records):
        raise ValueError('size must be >= base size')
    recs = list(base_records)
    seen = {x['src'] for x in recs}
    need = size - len(recs)
    v = 0
    while need > 0:
        for item in base_records:
            if need <= 0:
                break
            src_new = rewrite_with_template(item['src'], v)
            if src_new in seen:
                continue
            seen.add(src_new)
            recs.append({
                'src': src_new,
                'rephrase': build_rephrase(src_new),
                'alt': item['alt'],
                'loc': item.get('loc', item['src']),
                'loc_ans': item.get('loc_ans', ''),
            })
            need -= 1
        v += 1
    return recs

for size in [120, 180]:
    out = Path(f'/content/work/edits_bias_stress_{size}.json')
    recs = build_records(base, size)
    out.write_text(json.dumps(recs, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved:', out, 'count=', len(recs), 'unique_src=', len({x['src'] for x in recs}))


In [ ]:
# Purpose: Choose expanded edit size and run settings for this notebook.

EDIT_SIZE = 120   # Change to 180 for larger edit injection.
ROUNDS = 10
STEP = max(1, EDIT_SIZE // ROUNDS)

assert EDIT_SIZE in (120, 180), 'EDIT_SIZE must be 120 or 180'
print({'EDIT_SIZE': EDIT_SIZE, 'ROUNDS': ROUNDS, 'STEP': STEP})


In [ ]:
# Purpose: Generate ROME config for GPT-2 (full-eval, expanded edits).

import yaml
from pathlib import Path

src = Path('/content/EasyEdit/hparams/ROME/gpt2-xl.yaml')
dst = Path('/content/work/ROME_gpt2_full_expanded.yaml')

cfg = yaml.safe_load(src.read_text(encoding='utf-8'))
cfg['alg_name'] = 'ROME'
cfg['model_name'] = 'gpt2'
cfg['device'] = 0
cfg['layers'] = [9, 10, 11]
cfg['v_loss_layer'] = 11
cfg['v_weight_decay'] = cfg.get('v_weight_decay', 0.5)
cfg['mom2_dataset'] = 'wikitext'
cfg['mom2_n_samples'] = 300
cfg['mom2_dtype'] = 'float32'
cfg['stats_dir'] = '/content/work/cache/stats'
cfg['fact_token'] = cfg.get('fact_token', 'subject_last')
cfg['model_parallel'] = False
cfg['fp16'] = False

Path('/content/work/cache/stats').mkdir(parents=True, exist_ok=True)

dst.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('written:', dst)
print({k: cfg.get(k) for k in ['alg_name','model_name','layers','v_loss_layer','mom2_dataset','mom2_n_samples','mom2_dtype','v_weight_decay','stats_dir']})


In [ ]:
# Purpose: Run ROME(gpt2) with full evaluation and expanded edit set.

import os, re, glob, subprocess

METHOD = 'ROME'
HPARAMS = '/content/work/ROME_gpt2_full_expanded.yaml'
EDITS_JSON = f'/content/work/edits_bias_stress_{EDIT_SIZE}.json'
CROWS_JSON = '/content/work/crows_pairs_full.jsonl'
BBQ_JSON = '/content/work/bbq_full.jsonl'

RUN_TAG = f"{METHOD}_gpt2_full_e{EDIT_SIZE}"
OUT_ROOT = f"/content/work/results/{METHOD}/{RUN_TAG}"
OUT_DIR = f"{OUT_ROOT}/rounds"
os.makedirs(OUT_DIR, exist_ok=True)

CLEAN_START = False
if CLEAN_START and os.path.exists(OUT_ROOT):
    import shutil
    shutil.rmtree(OUT_ROOT, ignore_errors=True)
    os.makedirs(OUT_DIR, exist_ok=True)

round_files = glob.glob(os.path.join(OUT_DIR, 'round_*.json'))
max_round = -1
for p in round_files:
    m = re.search(r'round_(\d+)\.json$', p)
    if m:
        max_round = max(max_round, int(m.group(1)))

start_round = max_round + 1 if max_round >= 0 else 0
print('existing max_round=', max_round, '=> start_round=', start_round)
print('HPARAMS=', HPARAMS)
print('EDITS_JSON=', EDITS_JSON)
print('OUT_DIR=', OUT_DIR)

cmd = [
    'python', '-u', 'run_edit_fairness_rounds.py',
    '--hparams', HPARAMS,
    '--edits_json', EDITS_JSON,
    '--crows', CROWS_JSON,
    '--bbq', BBQ_JSON,
    '--rounds', str(ROUNDS),
    '--step', str(STEP),
    '--start_round', str(start_round),
    '--out_dir', OUT_DIR,
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TOKENIZERS_PARALLELISM'] = 'false'
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
env['PYTHONPATH'] = '/content/EasyEdit:' + env.get('PYTHONPATH', '')

p = subprocess.Popen(cmd, cwd='/content/work', env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
ret = p.wait()
if ret != 0:
    raise RuntimeError(f'run failed: {ret}')


In [ ]:
# Purpose: Compute drift metrics, draw curve, and print key values.

import json, subprocess

DRIFT_JSON = f'{OUT_ROOT}/drift_metrics.json'
DRIFT_PNG = f'{OUT_ROOT}/drift_curve.png'

subprocess.run([
    'python', '/content/work/calc_drift_metrics.py',
    '--input_dir', OUT_DIR,
    '--pattern', 'round_*.json',
    '--out', DRIFT_JSON,
], check=True)

subprocess.run([
    'python', '/content/work/plot_drift.py',
    '--metrics_json', DRIFT_JSON,
    '--out_png', DRIFT_PNG,
], check=True)

print('saved:', DRIFT_JSON)
print('saved:', DRIFT_PNG)

d = json.load(open(DRIFT_JSON, 'r', encoding='utf-8'))
print('CDA =', d.get('cda'))
for r in d.get('rounds', []):
    print(f"round={r.get('round')} risk={r.get('fairness_risk')} fdr={r.get('fdr')} delta={r.get('delta_from_base')}")


In [ ]:
# Purpose: Zip outputs and download to local machine.

import os, subprocess
from google.colab import files

zip_name = f"{RUN_TAG}_rounds_and_metrics.zip"
subprocess.run(['bash', '-lc', f"cd '{OUT_ROOT}' && zip -r '{zip_name}' rounds drift_metrics.json drift_curve.png"], check=True)
zip_path = os.path.join(OUT_ROOT, zip_name)
print('zip =>', zip_path)
files.download(zip_path)
